Libraries

In [1]:
import pandas as pd
import os
from IPython.display import display

pd.set_option('display.max_columns', None)

In [2]:
# Definir rango de fechas para Enero 2021 (no está disponible todo enero 2020)

start_date = '2021-01-01'
end_date = '2021-01-31'
# end_date = '2022-12-31'

date_range = pd.date_range(start=start_date, end=end_date)
# print(date_range)

In [3]:
# Importar Enero 2021

df = []

for single_date in date_range:
    url = f'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/refs/heads/master/csse_covid_19_data/csse_covid_19_daily_reports/{single_date.strftime("%m-%d-%Y")}.csv'
    df.append(pd.read_csv(url))

# Combinar los DataFrames diarios en uno solo.
enero = pd.concat(df, ignore_index=True)

In [4]:
# 1. Cargar y visualizar los primeros 5 registros

enero.head()

,FIPS,Admin2,Province_State,Country_Region,Last_Update,Lat,Long_,Confirmed,Deaths,Recovered,Active,Combined_Key,Incident_Rate,Case_Fatality_Ratio
0,NaN,NaN,NaN,Afghanistan,2021-01-02 05:22:33,33.93911,67.709953,52513,2201,41727,8585,Afghanistan,134.896578,4.191343
1,NaN,NaN,NaN,Albania,2021-01-02 05:22:33,41.15330,20.168300,58316,1181,33634,23501,Albania,2026.409062,2.025173
2,NaN,NaN,NaN,Algeria,2021-01-02 05:22:33,28.03390,1.659600,99897,2762,67395,29740,Algeria,227.809861,2.764848
3,NaN,NaN,NaN,Andorra,2021-01-02 05:22:33,42.50630,1.521800,8117,84,7463,570,Andorra,10505.403482,1.034865
4,NaN,NaN,NaN,Angola,2021-01-02 05:22:33,-11.20270,17.873900,17568,405,11146,6017,Angola,53.452981,2.305328


In [5]:
# 2. Mostrar el número total de filas y columnas del DataFrame.

print('Filas en total: ', len(enero))
print('Columnas en total: ', len(enero.columns))

Filas en total:  124398
Columnas en total:  14


In [6]:
# 3. Describir los tipos de datos (dtypes) y convertir las columnas necesarias (por ejemplo,
# fechas).

enero.dtypes

FIPS                   float64
Admin2                  object
Province_State          object
Country_Region          object
Last_Update             object
Lat                    float64
Long_                  float64
Confirmed                int64
Deaths                   int64
Recovered                int64
Active                   int64
Combined_Key            object
Incident_Rate          float64
Case_Fatality_Ratio    float64
dtype: object

In [7]:
# 3

# Formatear la columna Last_Update a tipo datetime

enero['Last_Update'] = pd.to_datetime(enero['Last_Update'], format = 'ISO8601')

enero.dtypes

FIPS                          float64
Admin2                         object
Province_State                 object
Country_Region                 object
Last_Update            datetime64[ns]
Lat                           float64
Long_                         float64
Confirmed                       int64
Deaths                          int64
Recovered                       int64
Active                          int64
Combined_Key                   object
Incident_Rate                 float64
Case_Fatality_Ratio           float64
dtype: object

In [8]:
# 4. Detectar y mostrar valores nulos o faltantes por columna.

display(enero.isnull().sum())

FIPS                   23166
Admin2                 23011
Province_State          5529
Country_Region             0
Last_Update                0
Lat                     2776
Long_                   2776
Confirmed                  0
Deaths                     0
Recovered                  0
Active                     0
Combined_Key               0
Incident_Rate           2776
Case_Fatality_Ratio     1484
dtype: int64

In [9]:
# 5. Eliminar columnas irrelevantes (por ejemplo, códigos FIPS o coordenadas si no se usarán).

enero = enero.drop(columns=['FIPS', 'Admin2', 'Lat', 'Long_', 'Combined_Key'])
enero.head(1)

,Province_State,Country_Region,Last_Update,Confirmed,Deaths,Recovered,Active,Incident_Rate,Case_Fatality_Ratio
0,NaN,Afghanistan,2021-01-02 05:22:33,52513,2201,41727,8585,134.896578,4.191343


In [10]:
# 6. Estandarizar nombres de columnas (usar formato snake_case).

enero.columns = enero.columns.str.lower().str.replace(' ', '_')
enero.head(1)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio
0,NaN,Afghanistan,2021-01-02 05:22:33,52513,2201,41727,8585,134.896578,4.191343


In [11]:
# 7. Homogeneizar nombres de países (ej. “US” → “United States”).

enero['country_region'] = enero['country_region'].replace({'US': 'United States'})
enero[enero['country_region'] == 'United States'].head(1)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio
648,Alabama,United States,2021-01-02 05:22:33,4239,50,0,4189,7587.391935,1.179523


In [12]:
# 8. Convertir la columna last_update al formato YYYY-MM-DD (día preciso)
# Asegurar datetime y mantener sólo fecha (YYYY-MM-DD)
enero['last_update'] = pd.to_datetime(enero['last_update'], errors='coerce').dt.date
enero.head(1)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio
0,NaN,Afghanistan,2021-01-02,52513,2201,41727,8585,134.896578,4.191343


In [13]:
# 9. Crear una columna active_cases = Confirmed - Deaths - Recovered.

enero['active_cases'] = (enero['confirmed'] - enero['deaths'] - enero['recovered'])
enero.head(1)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio,active_cases
0,NaN,Afghanistan,2021-01-02,52513,2201,41727,8585,134.896578,4.191343,8585


In [14]:
# 10. Guardar el DataFrame limpio como covid_clean_enero2021.csv e indicar su tamaño en MB.

try:
    enero.to_csv('covid_clean_enero2021.csv')
except:
    os.remove('covid_clean_enero2021.csv')
    enero.to_csv('covid_clean_enero2021.csv')

file_size = os.path.getsize('covid_clean_enero2021.csv') / (1024 * 1024)  # Convertir a MB
print(f'El tamaño del archivo covid_clean_enero2021.csv es: {file_size:.2f} MB')

El tamaño del archivo covid_clean_enero2021.csv es: 11.20 MB
